# Introduction 

<font size="3">This notebook demonstrates the complete SunSCC inference pipeline for automated sunspot analysis. Starting from a white-light (WL) solar image in FITS format, you'll segment individual sunspots, cluster them into physically meaningful groups, and classify each group according to the McIntosh taxonomy. Beyond understanding the full pipeline architecture, you'll learn how to load pretrained models, run inference on new solar observations, and apply the clustering algorithm to group detected features.


<div style="width: 750px;  padding: 15px; border: 1px solid transparent; border-color: transparent; margin-bottom: 20px; border-radius: 4px; color: #8a6d3b;; background-color: #fcf8e3; border-color: #faebcc;">
This tutorial requires pretrained segmentation and classification models from the SunSCC package. Model training is covered separately in the repository's README.md. Ensure you have completed training or obtained pretrained weights before proceeding.
</div>
 </font>

# Big Picture

<font size="3">
The SunSCC pipeline transforms raw solar observations into structured, scientifically annotated data. The architecture mirrors the hierarchical nature of solar active region analysis, progressing from pixel-level detection to region-level classification.

After loading the pretrained segmentation network and preprocessing the input FITS image, inference generates a binary mask where activated pixels correspond to sunspot regions. Visualizing this output provides immediate feedback on detection quality and helps identify potential preprocessing issues or edge cases in the input data.

The clustering algorithm then operates on connected components within the predicted mask, leveraging spatial proximity and morphological properties to aggregate individual sunspots into coherent active regions. This step bridges the gap between low-level segmentation outputs and the region-level structure required for McIntosh classification.

Finally, each identified cluster is extracted and evaluated by the pretrained classifier, which assigns the appropriate McIntosh class based on the group's morphological characteristics. The pipeline output associates each detected active region with its corresponding classification, enabling downstream scientific analysis or cataloging.
</font>

# Code

<font size="3">Import required libraries and dependencies for the pipeline.</font>


In [ ]:
import os
from sunscc.nb.load import load_from_dir, load_from_dir2, load_from_dir_new_pl, load_from_cfg
from pathlib import Path
from astropy.io import fits
import skimage.io as io
import numpy as np
import torch 
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
%matplotlib ipympl

from sunscc.dataset import DeepsunSegmentationPredictDataset, get_grid_size, crop_patch
from sunscc.dataset.utils import *
from sunscc.callback import reconstruct_segmentation_image, display_classification_predictions

### Set environment variable and path to the image to be processed

<font size="3">Configure the SUNSCCDATASET_PATH environment variable to the repository root and specify the input image path and output directory for processed results.</font>

In [ ]:
cwd = Path(os.getcwd())
tmp = cwd.parents[0]
os.environ['SUNSCCDATASET_PATH'] = str(tmp)
print(os.environ['SUNSCCDATASET_PATH'])

input_image = "./UPH20240510080302.FTS"
output_dir = Path('./predictions')
if not output_dir.exists():
    output_dir.mkdir()

# Segmentation

<font size="3">Define path to pretrained segmentation model.</font>

In [ ]:
segmenter_run_dir = Path("/home/rwc/sunscc_repo/sunscc-rob/outputs/2025-02-25/19-29-57_2013-15_UNet_T425_T375_T325_StepLR_epoch_1_run0") #Path(os.environ['SUNSCCDATASET_PATH']) / 'outputs/SunSCC_Model' / 'Segmentation/seg_run0'
print(segmenter_run_dir)

<font size="3">Load the pretrained sunspot segmentation model from the specified checkpoint path and obtain the inference dataloader.</font>

In [ ]:
dataset_overrides = [
        'dataset.test_dataset.transforms=[]', 
    ]
seg_config, seg_model, seg_dm, seg_trainer = load_from_dir_new_pl(
                                    run_path= segmenter_run_dir.resolve(),
                                    # run_path= run_dir,
                                    load_trainer=False,
                                    override = dataset_overrides + [
                                        # Don't create new wandb run when testing/predicting
#                                         f'logger=[{{_target_: pytorch_lightning.loggers.CSVLogger, save_dir: "csv_metrics"}}]',
                                        '~callbacks.LR_monitor',
                                        #'~callbacks.wandb',
                                    ]
                                )
# dm.setup()
test_dataloader = seg_dm.test_dataloader()

### Load input image, split it in a grid and create a batch for segmentation model

<font size="3">Open the input FITS image, partition it into 512×512 pixel patches and construct a batched dataset where each element contains the image patch, sample identifier, filename, and segmentation placeholder.</font>

In [ ]:
hdulst:fits.HDUList = fits.open(input_image)
image = hdulst[0]
header = image.header
center = np.array(image.shape)//2
radius = header['SOLAR_R']

patch_side = test_dataloader.dataset.patch_side
grid_size = get_grid_size(input_image, patch_side)

batch_v1 = {}

for index in tqdm(range(grid_size)):

    sample = {} 
    idx_img = index // grid_size
    idx_patch = index % grid_size

    sample["image"] = crop_patch((image.data).astype(float), idx_patch, patch_side, grid_size) # load image from directory with skimage
    sample['sample_id'] = f'{idx_img}_{idx_patch}'
    sample["name"] = os.path.basename(input_image)

    sample["segmentation"] = np.zeros_like(sample["image"])
    
    batch_v1[idx_patch] = sample

hdulst.close()

<font size="3">Execute segmentation inference across all patches, reconstruct the full-disk prediction map through stitching, and persist the result to disk.</font>

In [ ]:
keys_to_tensor = ['image', 'segmentation']

batch_v2 = {key: torch.tensor(np.concatenate([d[key][None,:,:] for d in batch_v1.values()])) 
            if key in keys_to_tensor else [d[key] for d in batch_v1.values()] 
            for key in batch_v1[0].keys()}

with torch.no_grad():
    output_seg = seg_model.predict(batch_v2)
reconstruct_segmentation_image(grid_size, batch_v2, output_seg, output_dir)

###  Load segmentation mask and show

<font size="3">Load and visualize the full-disk segmentation map to assess detection quality and coverage.</font>

In [ ]:
seg_mask_fn = output_dir/(os.path.basename(input_image).split(".")[0]+ ".png")
seg_mask = io.imread(seg_mask_fn)

fig  = plt.figure(figsize=(8,4))
ax1 = plt.subplot(1, 1, 1)
ax1.imshow(image.data, cmap='gray', interpolation=None)
ax1.imshow(seg_mask, cmap='jet', alpha=0.5 )

# Clustering

<font size="3">Import utility functions for the clustering algorithm.</font>

In [ ]:
import sunscc.utils.clustering.clustering_utilities as c_utils

<font size="3">Define clustering algorithm parameters and apply the algorithm to connected components in the sunspot segmentation mask.### Define clustering parameters and find sunspots groups in image</font>

In [ ]:
wl_fn , seg_mask_fn = input_image, seg_mask_fn

input_type = 'mask'
Rmm = 126.0
look_distance=0.1
kernel_bandwidthLon=.35
# kernel_bandwidthLon=.45
kernel_bandwidthLat=0.08
n_iterations=20

clustering_key, clustering_dict, (m,h,wcs,seg) = c_utils.process_new_image(wl_fn , seg_mask_fn,
                                                            input_type=input_type, Rmm=Rmm,
                                                             look_distance=look_distance, n_iterations=n_iterations,
                                                             kernel_bandwidthLat=kernel_bandwidthLat,
                                                             kernel_bandwidthLon=kernel_bandwidthLon,
                                                            show=False
                                                            )

<font size="3">Visualize clustering results overlaid on the original FITS image.</font>

In [ ]:

colors = [ "red", "green", "blue", "orange", "purple", "brown", "pink", "gray", "olive", "cyan", "magenta", "yellow"] 

fig3,ax3 = plt.subplots(1,2,figsize=(10,4))

ax3[0].clear(), ax3[1].clear()#, ax3[2].clear()
ax3[0].set_title(input_image)
ax3[0].imshow(m.data,cmap='gray')
ax3[0].imshow(seg,cmap='jet',alpha=0.5)
ax3[0].invert_yaxis()

ax3[1].imshow(m.data,cmap='gray')
ax3[1].invert_yaxis()

ax3[1].set_title(f'angle: {"%.2f" % clustering_dict["SOLAR_P0"]}, should_flip: {clustering_dict["should_flip"]}')
if clustering_dict['seg'] is not {}:
    
    ms_group_sunspots = clustering_dict['meanshift']['groups']
    ms_group_sunspots_px =  clustering_dict['meanshift']['groups_px']
    ms_centers = clustering_dict['meanshift']['centroids_px']
    
    for i in range(len(ms_group_sunspots)):
        c = colors[i%len(colors)]
        cur = np.array(ms_group_sunspots_px[i])
        cur_center = np.array(ms_centers[i])
        
        ax3[1].scatter(cur_center[0],cur_center[1], s=10, c=c, marker='x')
        ax3[1].scatter(cur[:,0], cur[:,1], color=c, s=2)

# Classification

<font size="3">Import utility functions for the classification stage.</font>

In [ ]:
from sunscc.utils.clustering import matching_utilities as m_utils
from copy import deepcopy

from omegaconf import DictConfig, OmegaConf
from sunscc.dataset.transform.pipelines import Compose
from sunscc.dataset.utils import create_circular_mask, get_sun_mask

<font size="3">Specify the path to the pretrained McIntosh classification model.</font>

In [ ]:
# classifier_run_dir = Path('/Volumes/WB_EXT_Niels/ROB_SEMINAR/Models/Classification/classif_run0')
classifier_run_dir = Path(os.environ['SUNSCCDATASET_PATH']) / 'outputs/SunSCC_Model' / 'Classification/classif_run0'
classifier_ckpt_path = classifier_run_dir / 'models' / 'ENCODER_MLP1_MLP2_MLP3.ckpt'
print(classifier_run_dir)


<font size="3">Load the pretrained sunspot group classification model from the checkpoint and initialize the inference dataloader.</font>

In [ ]:
dataset_overrides = []
classif_config, classif_model, classif_dm, _ = load_from_dir2(
                                    run_path= classifier_run_dir.resolve(),
                                    # run_path= run_dir,
                                    model_path= classifier_ckpt_path.resolve(),
                                    load_trainer=False,
                                    override = dataset_overrides + [
                                        # Don't create new wandb run when testing/predicting
#                                         f'logger=[{{_target_: pytorch_lightning.loggers.CSVLogger, save_dir: "csv_metrics"}}]',
                                        f'logger=[]',
                                        '~callbacks.LR_monitor',
                                        '~callbacks.wandb',
                                    ]
                                )
# dm.setup()
test_dataloader = classif_dm.test_dataloader()

<font size="3">Retrieve the possible values for each McIntosh component based on the classification model's training configuration.</font>

In [ ]:
first_classes = classif_config.first_classes
second_classes = classif_config.second_classes
third_classes = classif_config.third_classes

c1_mapper = {c: i for i,c in enumerate(first_classes)}
c2_mapper = {c: i for i,c in enumerate(second_classes)}
c3_mapper = {c: i for i,c in enumerate(third_classes)}

def add_superclasses(group_dict):
    cpy = deepcopy(group_dict)    
    # print(cpy)

    cpy["SuperClass"] = {
        "1": "?",
        "2": "?",
        "3": "?",
    }
    
    return cpy

<font size="3">Extract group-centered patches from the input image and format group metadata into classification samples. 

Each sample contains pixel and Carrington coordinates of the group and its constituent sunspots, angular eccentricity, angular distance to disk center, and classification placeholders. </font>

In [ ]:

Rpx = h["SOLAR_R"]
# Rpx = 1480
bn , img_dict = m_utils.find_groups_one_image(wl_fn , seg_mask_fn,
                                            clustering_dict, 
                                            Rpx,
                                            input_type=input_type,
                                            
                                           )

groups = img_dict['groups']
group_based_dataset_superclasses = {}
for i, g in enumerate(groups):
    g['centroid_px'] = [g['centroid_px'][1],g['centroid_px'][0]]
    g['members'] = [[a[1],a[0]] for a in g['members']]
    g['members_mean_px'] = [g['members_mean_px'][1],g['members_mean_px'][0]]
    
    new_group_infos= {
        'angle': img_dict['angle'],
        'deltashapeX': img_dict['deltashapeX'],
        'deltashapeY':img_dict['deltashapeY'],
        'centroid_px': g['centroid_px'],
        'centroid_Lat': g['centroid_Lat'],
        'centroid_Lon': g['centroid_Lon'],
        'members': g['members'],
        'members_mean_px': g['members_mean_px'],
        'angular_excentricity_rad': g['angular_excentricity_rad'],
        'angular_excentricity_deg': g['angular_excentricity_deg'],
        'Zurich': g['Zurich'],
        'McIntosh': g['McIntosh'],   
    }

    new_goup_id = bn + '_' + str(i)
    
    group_based_dataset_superclasses[new_goup_id] = add_superclasses(new_group_infos)


<font size="3">The patches undergo preprocessing transforms that erase sunspots not belonging to the target group, preventing misclassification from neighboring active regions.

Each extracted and preprocessed patches that will serve as input to the classification model is visualized.</font>

In [ ]:
classification = 'SuperClass'

transforms_p1 = OmegaConf.load('../sunscc/conf/exp/Classification_Superclasses4.yaml').dataset.train_dataset.transforms
transforms_p1[3].standard_height = 350
transforms_p1[3].standard_width = 350

if isinstance(transforms_p1, collections.abc.Mapping):
    transforms_p1 = partial(call, config=transforms_p1)
elif isinstance(transforms_p1, collections.abc.Sequence):
    transforms_init = []
    for transform in transforms_p1:
        transforms_init.append(instantiate(transform))
    transforms = Compose(transforms_init)
    
classif_samples = {}

for index, (g_k, g_v) in tqdm(enumerate(group_based_dataset_superclasses.items())):
    group_dict = g_v
    basename = g_k.split('_')[0]
    
    sample = {} 
    
    basic_mask = create_circular_mask( image.shape[0], image.shape[1] ,center,radius=radius*1.03)
    sample['solar_disk'] = get_sun_mask( image.data, basic_mask, radius)
    
    sample['excentricity_map'] = create_excentricity_map(sample['solar_disk'], radius, value_outside=-1)
    sample['mask'] = io.imread(seg_mask_fn)
    sample['confidence_map'] = io.imread(seg_mask_fn)

    sample['image'] = (image.data).astype(float)

    sample['members'] = np.array(group_dict['members']) if 'members' in group_dict else np.array([0])
    sample['members_mean_px'] = np.array(group_dict['members_mean_px']) if 'members_mean_px' in group_dict else np.array([0])

    sample['name'] = basename
    sample['group_name'] = g_k

    sample['solar_angle'] = group_dict['angle']
    sample['deltashapeX'] = group_dict['deltashapeX']
    sample['deltashapeY'] = group_dict['deltashapeY']

    sample['angular_excentricity'] = np.array([group_dict["angular_excentricity_deg"]])
    sample['centroid_px'] = np.array(group_dict["centroid_px"])
    sample['centroid_Lat'] = np.array([group_dict["centroid_Lat"]])

    sample['class1'] = group_dict["SuperClass"]['1']
    sample['class2'] = group_dict["SuperClass"]['2']
    sample['class3'] = group_dict["SuperClass"]['3']
    
    if transforms is not None:
        sample = transforms(**sample)
        
    fig,ax = plt.subplots(1, 5, figsize=(10, 2) )
    ax[0].imshow(sample["image"], cmap='gray', interpolation='none')
    ax[0].set_title('Image')
    ax[1].imshow(sample["mask"], cmap='gray', interpolation='none', vmin=0, vmax=1)
    ax[0].set_title('mask')
    ax[2].imshow(sample["confidence_map"], cmap='gray', interpolation='none', vmin=0, vmax=1)
    ax[0].set_title('confidence_map')
    ax[3].imshow(sample["solar_disk"], cmap='gray', interpolation='none', vmin=0, vmax=1)
    ax[0].set_title('solar_disk')
    ax[4].imshow(sample["excentricity_map"], cmap='gray', interpolation='none', vmin=0, vmax=1)
    ax[0].set_title('excentricity_map')
    fig.tight_layout()
        
    center  = sample['image'].shape[0]//2 , sample['image'].shape[1]//2
    slice_x = center[0]-( 256 //2 ), center[0]+ (256 //2 )
    slice_y = center[1]-( 256 //2 ), center[1]+ (256 //2 )
    center_region = sample['confidence_map'][slice_x[0]:slice_x[1], slice_y[0]:slice_y[1]]
    if np.sum(center_region) == 0:
        print('skipping sample', index)
        sample = None
    
    if sample is not None:
        if sample['group_name'] not in classif_samples:
            classif_samples[sample['group_name']] = sample

    
    
    

<font size="3">Convert the list of samples into batched tensors suitable for model inference.</font>

In [ ]:
transforms_p2 = OmegaConf.load(classifier_run_dir/'.hydra'/'config.yaml').dataset.test_dataset.transforms

if isinstance(transforms_p2, collections.abc.Mapping):
    transforms_p2 = partial(call, config=transforms_p2)
elif isinstance(transforms_p2, collections.abc.Sequence):
    transforms_init = []
    for transform in transforms_p2:
        transforms_init.append(instantiate(transform))
    transforms = Compose(transforms_init)

batch_v0 = classif_samples
batch_v1 = {}
if transforms is not None:
    for k, sample in batch_v0.items():
        sample = transforms(**sample)
        batch_v1[k] = sample

batch_keys = list(batch_v1.keys())
sample_keys = list(batch_v1[batch_keys[0]].keys())

keys_to_tensor = ['image', 'excentricity_map', 'solar_disk', 'mask', 'group_mask', 'confidence_map', 'group_confidence_map']

batch_v2 = {key: torch.tensor(np.concatenate([d[key][None,None,:,:] for d in batch_v1.values()])) 
            if key in keys_to_tensor else [d[key] for d in batch_v1.values()] 
            for key in sample_keys}

<font size="3"> Execute McIntosh classification inference on all sunspot groups and visualize predictions overlaid on the corresponding image patches. </font>

In [ ]:
classif_batch = batch_v2

output_classif = classif_model.predict(classif_batch)
# print(output_classif)
display_classification_predictions(classif_batch, output_classif, c1_mapper,c2_mapper,c3_mapper)
# print(output_classif)